# Démo 6 — Appels d'outils

**Contexte :** assistant du service client de Lagune Banque.

Le modèle ne connaît pas l'état des systèmes de la banque. Les appels d'outils lui permettent de demander au code une opération, comme consulter le statut d'un virement ou les horaires du service client. Le code exécute l'outil et transmet son résultat au modèle.

**Sommaire**
- [Configuration](#configuration)
- [Définition des outils](#outils)
- [Demande d'appel](#appel-outil)
- [Exécution de l'outil](#execution-outil)
- [Boucle d'appels](#boucle)
- [À retenir](#a-retenir)

```mermaid
sequenceDiagram
    actor Client
    participant Code as Code
    participant LLM as LLM

    Client->>Code: Où en est mon virement VIR-48213 ?
    Code->>LLM: question + description des outils
    LLM-->>Code: demande d'appel : check_transaction("VIR-48213")
    Note over Code: valide les arguments,<br/>exécute la fonction
    Code->>LLM: résultat de l'outil (rôle tool)
    LLM-->>Code: réponse rédigée
    Code->>Client: Votre virement de 25 000 F est en attente
```

## <a id="configuration"></a>Configuration

In [ ]:
import json
import os
import re
import sys
import time
from pathlib import Path

import litellm
import pandas as pd

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage"), Path("../../../code_partage"), Path("../../../../code_partage")] if (p / "config.py").exists()).resolve()))
from config import prepare

In [ ]:
prepare()

In [ ]:
SEED = 42


def ask(messages, **options):
    debut = time.perf_counter()
    r = litellm.completion(model=options.pop("model", os.environ["MODELE"]), api_key=os.environ["LLM_API_KEY"], seed=SEED, num_retries=4, messages=messages, **options)
    return {"text": (r.choices[0].message.content or "").strip(), "in": r.usage.prompt_tokens, "out": r.usage.completion_tokens, "seconds": time.perf_counter() - debut, "raw": r}

## <a id="outils"></a>Définition des outils

Les fonctions sont du code Python. Leur description, transmise au modèle avec les paramètres attendus, l'aide à choisir un outil. Cette démo définit un outil de suivi des virements et un outil de consultation des horaires.

In [ ]:
TRANSACTIONS = {
    "VIR-48213": {"statut": "en attente", "montant_fcfa": 25000, "date": "02/10/2026"},
    "VIR-51007": {"statut": "exécuté", "montant_fcfa": 120000, "date": "05/10/2026"},
}


def check_transaction(reference: str) -> dict:
    return TRANSACTIONS.get(reference, {"erreur": f"référence {reference} introuvable"})


def get_opening_hours() -> dict:
    return {"lundi_vendredi": "8h à 17h", "samedi": "9h à 13h", "dimanche": "fermé"}


TOOLS = [{"type": "function", "function": {
    "name": "check_transaction",
    "description": "Donne le statut, le montant et la date d'un virement Lagune Money à partir de sa référence. À utiliser dès que le client cite une référence VIR-xxxxx.",
    "parameters": {"type": "object", "properties": {"reference": {"type": "string", "description": "Référence du virement, au format VIR-12345"}}, "required": ["reference"]}}},
{"type": "function", "function": {
    "name": "get_opening_hours",
    "description": "Donne les horaires d'ouverture du service client de Lagune Banque. À utiliser quand le client demande quand il peut joindre ou rencontrer un conseiller.",
    "parameters": {"type": "object", "properties": {}}}}]

## <a id="appel-outil"></a>Demande d'appel

La question et la description de l'outil sont envoyées au modèle. S'il choisit cet outil, il renvoie une demande structurée avec le nom de la fonction et ses arguments. À ce stade, le code n'a encore rien exécuté.

In [ ]:
SYSTEME = {"role": "system", "content": "Tu es l'assistant du service client de Lagune Banque. Réponds en deux phrases au plus."}
messages = [SYSTEME, {"role": "user", "content": "Où en est mon virement VIR-48213 ?"}]
reponse = ask(messages, tools=TOOLS)["raw"].choices[0].message
appel = reponse.tool_calls[0]
print("contenu texte :", repr(reponse.content))
print("fonction demandée :", appel.function.name)
print("arguments (JSON) :", appel.function.arguments)

## <a id="execution-outil"></a>Exécution de l'outil

Le code vérifie le nom de l'outil et ses arguments, exécute la fonction autorisée, puis renvoie le résultat au modèle dans un message de rôle `tool`. Le modèle peut alors formuler une réponse au client.

In [ ]:
messages.append({"role": "assistant", "content": reponse.content, "tool_calls": [{"id": appel.id, "type": "function", "function": {"name": appel.function.name, "arguments": appel.function.arguments}}]})
resultat = check_transaction(**json.loads(appel.function.arguments))
print("résultat de la fonction :", resultat)
messages.append({"role": "tool", "tool_call_id": appel.id, "content": json.dumps(resultat, ensure_ascii=False)})
print("\nréponse finale :", ask(messages, tools=TOOLS)["text"])

## <a id="boucle"></a>Boucle d'appels

La boucle complète répète ce processus tant que le modèle demande un outil. `FONCTIONS` définit les fonctions autorisées ; le code n'exécute que celles qui y figurent.

In [ ]:
FONCTIONS = {"check_transaction": check_transaction, "get_opening_hours": get_opening_hours}
JOURNAL = []


def assistant(question):
    messages = [SYSTEME, {"role": "user", "content": question}]
    while True:
        message = ask(messages, tools=TOOLS)["raw"].choices[0].message
        if not message.tool_calls:
            return message.content.strip()
        messages.append({"role": "assistant", "content": message.content, "tool_calls": [{"id": t.id, "type": "function", "function": {"name": t.function.name, "arguments": t.function.arguments}} for t in message.tool_calls]})
        for t in message.tool_calls:
            args = json.loads(t.function.arguments)
            JOURNAL.append((t.function.name, args))
            resultat = FONCTIONS[t.function.name](**args) if t.function.name in FONCTIONS else {"erreur": "fonction inconnue"}
            messages.append({"role": "tool", "tool_call_id": t.id, "content": json.dumps(resultat, ensure_ascii=False)})


QUESTIONS = ["Où en est mon virement VIR-48213 ?", "Et le VIR-99999 ?", "Quels sont vos horaires d'ouverture ?", "Mon virement n'est pas arrivé, que faire ?"]
for q in QUESTIONS:
    avant = len(JOURNAL)
    reponse = assistant(q)
    print(f"{q}\n  appels : {JOURNAL[avant:] or 'aucun'}\n  réponse : {reponse[:160]}\n")

## <a id="a-retenir"></a>À retenir

- Le modèle propose un appel ; le code contrôle et exécute l'opération.
- Les descriptions et les paramètres des outils guident le choix du modèle.
- Le code doit valider les arguments et limiter l'exécution aux fonctions autorisées.
- Si une question manque d'informations, le modèle devrait demander une précision plutôt que d'inventer des arguments.